In [1]:
import torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
m = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    device_map="auto",
    torch_dtype="auto",
)
dm = AutoModelForCausalLM.from_pretrained(
    "DeepSeek-R1-Distill-Qwen-7B-W8A8-Dynamic-Per-Token",
    device_map="auto",
    torch_dtype="auto",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.14it/s]


In [5]:
pm = {k:v for k, v in m.named_parameters()}
pdm = {k:v for k, v in dm.named_parameters()}

In [9]:
pm.keys()

dict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.post_a

In [10]:
pdm.keys()

dict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.q_proj.weight_scale', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.k_proj.weight_scale', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.v_proj.weight_scale', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.o_proj.weight_scale', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight_scale', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight_scale', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight_scale', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.q_proj.weight_scale', 'model.layers.1.self_attn.q_proj.weight', 'model.laye

In [16]:
pm['model.layers.0.self_attn.q_proj.weight']

Parameter containing:
tensor([[-9.7752e-06, -1.9165e-02,  1.2024e-02,  ..., -4.2725e-02,
         -7.6294e-03,  2.2125e-03],
        [ 1.2085e-02, -5.9326e-02,  9.8877e-03,  ..., -5.2246e-02,
          5.6763e-03,  2.7924e-03],
        [-1.0803e-02,  1.0681e-02,  3.9673e-03,  ..., -2.6733e-02,
         -1.0803e-02,  1.2939e-02],
        ...,
        [ 9.4604e-03,  9.8877e-03,  4.3945e-02,  ..., -3.2959e-02,
         -1.0498e-02, -5.0354e-03],
        [ 8.2397e-03, -2.5146e-02, -8.5449e-04,  ..., -3.3936e-02,
         -2.8809e-02,  1.2817e-02],
        [ 3.5645e-02, -2.1362e-02, -8.0490e-04,  ..., -3.1494e-02,
          1.6357e-02, -2.1362e-02]], device='cuda:0', dtype=torch.bfloat16,
       requires_grad=True)

In [15]:
pdm['model.layers.0.self_attn.q_proj.weight']

Parameter containing:
tensor([[  0, -10,   7,  ..., -21,  -4,   1],
        [  3, -14,   2,  ..., -12,   1,   1],
        [ -3,   3,   1,  ...,  -6,  -3,   3],
        ...,
        [  6,   7,  32,  ..., -21,  -7,  -2],
        [  8, -24,  -1,  ..., -30, -28,  11],
        [ 26, -16,  -1,  ..., -22,  12, -14]], device='cuda:0',
       dtype=torch.int8)

In [17]:
pdm['model.layers.0.self_attn.q_proj.weight'] / pm['model.layers.0.self_attn.q_proj.weight']

tensor([[  -0.,  520.,  584.,  ...,  492.,  524.,  452.],
        [ 248.,  236.,  202.,  ...,  230.,  176.,  358.],
        [ 278.,  280.,  252.,  ...,  224.,  278.,  232.],
        ...,
        [ 636.,  708.,  728.,  ...,  636.,  668.,  398.],
        [ 972.,  956., 1168.,  ...,  884.,  972.,  860.],
        [ 728.,  748., 1240.,  ...,  700.,  732.,  656.]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<DivBackward0>)

### least square with zero alpha

In [42]:
beta = (pdm['model.layers.0.self_attn.q_proj.weight'] * pm['model.layers.0.self_attn.q_proj.weight']).sum(dim=-1, keepdim=True).float() / (pm['model.layers.0.self_attn.q_proj.weight'] ** 2).sum(dim=-1, keepdim=True)

In [43]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta).int()

tensor([[  0, -10,   6,  ..., -22,  -4,   1],
        [  2, -14,   2,  ..., -12,   1,   0],
        [ -2,   2,   0,  ...,  -6,  -2,   3],
        ...,
        [  6,   6,  28,  ..., -21,  -6,  -3],
        [  7, -22,   0,  ..., -30, -26,  11],
        [ 25, -15,   0,  ..., -22,  11, -15]], device='cuda:0',
       dtype=torch.int32)

In [44]:
beta

tensor([[524.7205],
        [243.2000],
        [249.8313],
        ...,
        [651.3778],
        [908.3871],
        [715.6685]], device='cuda:0', grad_fn=<DivBackward0>)

## using default scale

In [8]:
pm['model.layers.0.mlp.gate_proj.weight'].size()

torch.Size([18944, 3584])

In [9]:
pm['model.layers.0.mlp.up_proj.weight'].size()

torch.Size([18944, 3584])

In [10]:
pm['model.layers.0.mlp.down_proj.weight'].size()

torch.Size([3584, 18944])

In [11]:
pdm['model.layers.0.mlp.gate_proj.weight_scale'].size()

torch.Size([18944, 1])

In [14]:
pdm['model.layers.0.mlp.gate_proj.weight_scale'][1]

tensor([0.0013], device='cuda:0', dtype=torch.bfloat16)

In [16]:
pdm['model.layers.0.mlp.gate_proj.weight'][1][1]

tensor(3, device='cuda:0', dtype=torch.int8)

In [17]:
pm['model.layers.0.mlp.gate_proj.weight'][1][1]

tensor(0.0022, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>)

In [21]:
pm['model.layers.0.post_attention_layernorm.weight'][1]

tensor(0.2256, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>)

In [22]:
pdm['model.layers.0.post_attention_layernorm.weight'][1]

tensor(0.1348, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>)

In [27]:
sq_cs = pm['model.layers.0.mlp.gate_proj.weight'][1][1] * pm['model.layers.0.post_attention_layernorm.weight'][1] / pdm['model.layers.0.post_attention_layernorm.weight'][1]

In [28]:
sq_cs

tensor(0.0038, device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [29]:
sq_cs / pdm['model.layers.0.mlp.gate_proj.weight_scale'][1]

tensor([2.9531], device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [30]:
input_scale = pm['model.layers.0.post_attention_layernorm.weight'] / pdm['model.layers.0.post_attention_layernorm.weight']

In [33]:
input_scale.size()

torch.Size([3584])

In [37]:
manual = pm['model.layers.0.mlp.gate_proj.weight'] * input_scale.view(1, -1) / pdm['model.layers.0.mlp.gate_proj.weight_scale'].view(-1, 1)

In [41]:
(torch.round(manual) - pdm['model.layers.0.mlp.gate_proj.weight'])

tensor([[-0.,  0.,  0.,  ...,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  1.,  0.],
        [ 0.,  0.,  0.,  ...,  1.,  0.,  0.],
        ...,
        [ 0., -0., -0.,  ..., -1., -1., -1.],
        [ 0.,  0.,  0.,  ...,  1.,  0., -1.],
        [ 0.,  0.,  0.,  ...,  1.,  0.,  0.]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SubBackward0>)

In [42]:
(manual.to(torch.int8) - pdm['model.layers.0.mlp.gate_proj.weight'])

tensor([[ 0,  1,  0,  ...,  0,  0,  0],
        [-1, -1, -1,  ...,  0,  0, -1],
        [-1,  0, -1,  ...,  0,  0,  1],
        ...,
        [ 0,  0,  0,  ..., -1, -1, -1],
        [ 0,  0,  1,  ...,  0,  0, -1],
        [-1, -1,  1,  ...,  0,  1,  0]], device='cuda:0', dtype=torch.int8)

In [48]:
mask = (torch.round(manual).to(torch.int8) - pdm['model.layers.0.mlp.gate_proj.weight']) != 0

In [49]:
torch.round(manual)[mask]

tensor([  8.,  -8., -38.,  ...,  58.,  24.,  24.], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<IndexBackward0>)

In [50]:
pdm['model.layers.0.mlp.gate_proj.weight'][mask]

tensor([  9,  -9, -39,  ...,  59,  25,  23], device='cuda:0', dtype=torch.int8)

In [52]:
manual[mask]

tensor([  8.5000,  -8.5000, -38.5000,  ...,  58.0000,  24.2500,  23.7500],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<IndexBackward0>)

## least square with alpha & beta

In [40]:
pdmw = pdm['model.layers.0.self_attn.q_proj.weight'].float()
pmw = pm['model.layers.0.self_attn.q_proj.weight']

pdmw = pdmw - pdmw.mean(dim=-1, keepdim=True)
pmw = pmw - pmw.mean(dim=-1, keepdim=True)

beta = (pdmw * pmw).sum(dim=-1, keepdim=True) / (pmw **2).sum(dim=-1, keepdim=True)
alpha = (pdm['model.layers.0.self_attn.q_proj.weight'] - pm['model.layers.0.self_attn.q_proj.weight'] * beta).mean(dim=-1, keepdim=True)

In [41]:
beta

tensor([[524.4738],
        [243.2699],
        [249.9417],
        ...,
        [652.8514],
        [908.6208],
        [715.0206]], device='cuda:0', grad_fn=<DivBackward0>)

In [45]:
alpha

tensor([[-0.0275],
        [-0.0240],
        [-0.0186],
        ...,
        [ 0.0100],
        [-0.0161],
        [-0.0866]], device='cuda:0', grad_fn=<MeanBackward1>)

In [47]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta + alpha).int() - pdm['model.layers.0.self_attn.q_proj.weight']

tensor([[ 0,  0, -1,  ..., -1,  0,  0],
        [-1,  0,  0,  ...,  0,  0, -1],
        [ 1, -1, -1,  ...,  0,  1,  0],
        ...,
        [ 0, -1, -4,  ...,  0,  1, -1],
        [-1,  2,  1,  ...,  0,  2,  0],
        [-1,  1,  1,  ...,  0, -1, -1]], device='cuda:0', dtype=torch.int32)

In [49]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta + alpha).int()

tensor([[  0, -10,   6,  ..., -22,  -4,   1],
        [  2, -14,   2,  ..., -12,   1,   0],
        [ -2,   2,   0,  ...,  -6,  -2,   3],
        ...,
        [  6,   6,  28,  ..., -21,  -6,  -3],
        [  7, -22,   0,  ..., -30, -26,  11],
        [ 25, -15,   0,  ..., -22,  11, -15]], device='cuda:0',
       dtype=torch.int32)

## Analyses

In [42]:
om = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    device_map="auto",
    torch_dtype="auto",
)

/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 32.50it/s]


In [45]:
profile = torch.load("./profile/qwen7b-r1.pt")

/tmp/ipykernel_11450/515699196.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  profile = torch.load("./profile/qwen7b-r1.pt")


In [ ]:
m = AutoModelForCausalLM.from_pretrained(
    "DeepSeek-R1-Distill-Qwen-7B-W8A8-Dynamic-Per-Token",
    device_map="auto",
    torch_dtype="auto",
)
mf = AutoModelForCausalLM.from_pretrained(
    "DeepSeek-R1-Distill-Qwen-7B-W8A8-onthefly-stable",
    device_map="auto",
    torch_dtype="auto",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  7.61it/s]


In [43]:
opm = {k:v for k, v in om.named_parameters()}

In [ ]:
pm = {k:v for k, v in m.named_parameters()}
pmf = {k:v for k, v in mf.named_parameters()}

In [22]:
pm['model.layers.0.self_attn.q_proj.weight'].float().norm(dim=0)

tensor([1121.5721, 1117.6010, 1187.6855,  ..., 1063.4374, 1145.2000,
        1092.9995])

In [19]:
pmf['model.layers.0.self_attn.q_proj.weight'].float().norm(dim=0)

tensor([1086.3379, 1049.6371, 1068.9874,  ..., 1074.6544, 1076.1111,
        1088.7612])

In [20]:
(pm['model.layers.0.self_attn.q_proj.weight'].float() - pmf['model.layers.0.self_attn.q_proj.weight'].float()).norm(dim=0)

tensor([ 50.5964,  77.4468, 124.0121,  ...,  47.0425,  83.2947,  47.4236])

In [24]:
dff = (pm['model.layers.0.self_attn.q_proj.weight'].float() - pmf['model.layers.0.self_attn.q_proj.weight'].float()).abs()

In [28]:
v, i = dff.max(dim=0)

In [31]:
v.max(dim=0)

torch.return_types.max(
values=tensor(255.),
indices=tensor(511))

In [32]:
v[511]

tensor(255.)

In [34]:
i[511]

tensor(2853)

In [39]:
dff[2853, 511]

tensor(255.)

In [40]:
pm['model.layers.0.self_attn.q_proj.weight'][2853, 511]

tensor(-128, dtype=torch.int8)

In [41]:
pmf['model.layers.0.self_attn.q_proj.weight'][2853, 511]

tensor(127, dtype=torch.int8)

In [44]:
opm['model.layers.0.self_attn.q_proj.weight'][2853, 511]

tensor(-0.1709, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [48]:
profile['model.layers.0.self_attn.q_proj.weight']

{'beta': tensor([[523.2613],
         [243.0839],
         [249.5435],
         ...,
         [652.6318],
         [908.0470],
         [714.3367]], requires_grad=True),
 'alpha': tensor([[ 0.1822],
         [ 0.0594],
         [ 0.0022],
         ...,
         [ 0.0449],
         [ 0.0446],
         [-0.5826]], requires_grad=True),
 'type': torch.int8}

In [49]:
res = opm['model.layers.0.self_attn.q_proj.weight'] * profile['model.layers.0.self_attn.q_proj.weight']['beta'] + profile['model.layers.0.self_attn.q_proj.weight']['alpha']

In [66]:
rd = torch.round(res)

In [56]:
res[2853, 511].int().to(profile['model.layers.0.self_attn.q_proj.weight']['type'])

tensor(127, dtype=torch.int8)

In [67]:
(rd - pm['model.layers.0.self_attn.q_proj.weight']).norm()

tensor(11103.4980, grad_fn=<LinalgVectorNormBackward0>)

In [69]:
pm['model.layers.0.self_attn.q_proj.weight'].float().norm()

tensor(66294.4141)

In [70]:
(rd - pm['model.layers.0.self_attn.q_proj.weight']).max()

tensor(110., grad_fn=<MaxBackward1>)

In [79]:
qres = res.clamp(min=-128, max=127).to(torch.int8)

In [80]:
(qres-pm['model.layers.0.self_attn.q_proj.weight']).float().abs().max()

tensor(110.)

In [83]:
(qres-pm['model.layers.0.self_attn.q_proj.weight']).float().norm()

tensor(11210.0664)

In [84]:
qres.float().norm()

tensor(64133.4922)

In [90]:
nd = qres-pm['model.layers.0.self_attn.q_proj.weight']

In [93]:
nd.max(dim=1)

torch.return_types.max(
values=tensor([ 43,  53, 109,  ...,  47,  87, 107], dtype=torch.int8),
indices=tensor([  32,   32,  662,  ..., 3322,  809,  662]))

In [96]:
nd[2].max(dim=0)

torch.return_types.max(
values=tensor(109, dtype=torch.int8),
indices=tensor(662))

In [97]:
nd[2, 662]

tensor(109, dtype=torch.int8)

In [98]:
qres[2, 662]

tensor(-19, dtype=torch.int8)

In [99]:
pm['model.layers.0.self_attn.q_proj.weight'][2, 662]

tensor(-128, dtype=torch.int8)

In [100]:
opm['model.layers.0.self_attn.q_proj.weight'][2, 662]

tensor(-0.0781, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [101]:
res[2, 662]

tensor(-19.4934, grad_fn=<SelectBackward0>)